<a href="https://colab.research.google.com/github/ridoy1211/Flyrank-Internship-ML/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ridoy1211/Flyrank-Internship-ML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
# ============================================
# SETUP
# ============================================

%pip -q install duckdb huggingface_hub

import os
import getpass
import duckdb
import pandas as pd
import numpy as np

# Get Hugging Face token from Colab Secret
HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN was not found. Add your Hugging Face READ token "
        "to Colab Secrets with the name HF_TOKEN."
    )

# DuckDB connection
con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}

print("Setup complete.")
print("HF token loaded:", bool(HF_TOKEN))

Setup complete.
HF token loaded: True


In [3]:
# ============================================
# Basic warehouse check
# ============================================

print(
    con.sql(f"""
        SELECT
            COUNT(*) AS n,
            MIN(report_date) AS min_date,
            MAX(report_date) AS max_date
        FROM {TABLES["fact_daily"]}
        WHERE month = '2026-03'
    """).df()
)

print("\nMarch 2026 rows checked.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

         n   min_date   max_date
0  9841378 2026-03-01 2026-03-31

March 2026 rows checked.


## 1. My rule and its reason codes

### Signal checks before the rule

I will test two signals before encoding the baseline:

1. **CTR vs. average position** — linked to FlyRank's CTR-fix logic.
2. **Search volume** — linked to the quick-win / volume idea.

I will report the evidence using bucket tables with `n` and give each signal an honest verdict:
`CONFIRMED`, `OPPOSITE`, `MIXED`, or `FALSE`.

### Proposed baseline rule

The baseline will prioritize pages that have meaningful search visibility but show a weaker-than-expected CTR for their position.

The rule will remain deliberately simple and hand-written. It will use no fitted model weights and no future-window or label-derived inputs.

Each ranked item will receive:

- a numeric `score`;
- one `reason_code`;
- one `action` label.

The exact threshold will be chosen from the observed signal distributions below rather than assumed in advance.

In [4]:
# ============================================
# Inspect fields used by the baseline
# ============================================

print(
    con.sql(f"""
        DESCRIBE
        SELECT *
        FROM {TABLES["dim_content"]}
    """).df()[["column_name", "column_type"]]
)

print("\nDaily performance fields:")

print(
    con.sql(f"""
        DESCRIBE
        SELECT *
        FROM {TABLES["fact_daily"]}
    """).df()[["column_name", "column_type"]]
)


                   column_name column_type
0               client_hash_id     VARCHAR
1              content_hash_id     VARCHAR
2              keyword_hash_id     VARCHAR
3                  url_hash_id     VARCHAR
4           keyword_char_count      BIGINT
5          keyword_token_count      BIGINT
6               url_char_count      BIGINT
7         content_created_date        DATE
8         content_updated_date        DATE
9                 content_type     VARCHAR
10               search_volume      BIGINT
11                 competition      DOUBLE
12           competition_level     VARCHAR
13                         cpc      DOUBLE
14                 main_intent     VARCHAR
15                   backlinks      BIGINT
16              category_count      BIGINT
17        keyword_created_date        DATE
18               provider_used     VARCHAR
19                  model_used     VARCHAR
20                  char_count      BIGINT
21                  word_count      BIGINT
22         

### Signal 1 — CTR vs. average position

**FlyRank flag connection:** CTR-fix logic.

**Question:** Do pages with stronger average search positions show meaningfully different CTR from pages with weaker positions?

I will aggregate March 2026 GSC performance to the content-page level and compare CTR across position buckets.

This is a signal audit only. I will not use the March outcome as a prediction label.

In [7]:
# ============================================
# Signal 1 — March page-level GSC metrics
# ============================================

page_march = con.sql(f"""
    SELECT
        f.client_hash_id,
        f.content_hash_id,

        SUM(f.gsc_impressions) AS impressions,
        SUM(f.gsc_clicks) AS clicks,

        CASE
            WHEN SUM(f.gsc_impressions) > 0
            THEN SUM(f.gsc_sum_position) / SUM(f.gsc_impressions)
            ELSE NULL
        END AS avg_position

    FROM {TABLES["fact_daily"]} f

    WHERE f.month = '2026-03'
      AND f.gsc_data_available = TRUE

    GROUP BY
        f.client_hash_id,
        f.content_hash_id

    HAVING SUM(f.gsc_impressions) > 0
""").df()

page_march["ctr"] = (
    page_march["clicks"] /
    page_march["impressions"] * 100
)

print(f"Page-level rows: {len(page_march):,}")

display(page_march.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Page-level rows: 176,738


,client_hash_id,content_hash_id,impressions,clicks,avg_position,ctr
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,6.893301,0.107313
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,3.214128,0.000000
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,6.535346,0.106572
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,7.435680,0.262945
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,429.0,1.0,3.871795,0.233100


In [6]:
# ============================================
# Signal 1 — Position buckets
# ============================================

page_march["position_bucket"] = pd.cut(
    page_march["avg_position"],
    bins=[0, 3, 10, 20, 50, np.inf],
    labels=["1-3", "4-10", "11-20", "21-50", "51+"],
    include_lowest=True
)

position_signal = (
    page_march
    .groupby("position_bucket", observed=False)
    .agg(
        mean_ctr=("ctr", "mean"),
        median_ctr=("ctr", "median"),
        impressions=("impressions", "sum"),
        clicks=("clicks", "sum"),
        n=("content_hash_id", "count")
    )
    .reset_index()
)

display(position_signal)

,position_bucket,mean_ctr,median_ctr,impressions,clicks,n
0,1-3,1.169594,0.0,41420140.0,160562.0,18860
1,4-10,0.487282,0.0,148112436.0,481189.0,83288
2,11-20,0.328465,0.0,31191659.0,98488.0,29922
3,21-50,0.237885,0.0,57614383.0,80635.0,32240
4,51+,0.084632,0.0,2318971.0,958.0,12428


In [8]:
# ============================================
# Signal 1 — Honest verdict helper
# ============================================

ordered_ctr = (
    position_signal
    .dropna(subset=["mean_ctr"])
    ["mean_ctr"]
    .to_numpy()
)

if len(ordered_ctr) >= 3:
    decreases = np.sum(np.diff(ordered_ctr) < 0)
    increases = np.sum(np.diff(ordered_ctr) > 0)

    if decreases >= len(ordered_ctr) - 1:
        signal1_verdict = "CONFIRMED"
    elif increases >= len(ordered_ctr) - 1:
        signal1_verdict = "OPPOSITE"
    else:
        signal1_verdict = "MIXED"
else:
    signal1_verdict = "FALSE"

print("Signal 1 verdict:", signal1_verdict)

Signal 1 verdict: CONFIRMED


### Signal 1 verdict

The verdict above is based on the observed direction of mean CTR across the position buckets.

I treat this as an empirical check rather than assuming that position must behave in a particular way. The bucket counts (`n`) are shown above so that very small groups are visible when interpreting the pattern.

### Signal 2 — Search volume

**FlyRank flag connection:** quick-win / volume logic.

**Question:** Does higher search volume correspond to stronger search visibility in this March slice?

Search volume comes from `dim_content` and is treated as a static content property. It is not a future performance label.

I will bucket search volume and inspect impressions, CTR, and the number of pages in each bucket.

In [14]:
# ============================================
# Signal 2 — Add search volume
# ============================================

volume_data = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        search_volume,
        content_type,
        main_intent
    FROM {TABLES["dim_content"]}
""").df()

# Ensure 'search_volume' exists in volume_data after SQL query
# This check is defensive against unexpected data retrieval issues.
if 'search_volume' not in volume_data.columns:
    print("Warning: 'search_volume' column missing from volume_data. Adding as NaN.")
    volume_data['search_volume'] = np.nan # Add with NaNs to prevent KeyError downstream

page_march = page_march.merge(
    volume_data,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

# After merge, ensure the column is there for display, even if all NaNs.
# This handles cases where the merge itself might unexpectedly drop columns (unlikely for left merge)
# or if 'search_volume' was only in some rows of volume_data, leading to unexpected behavior.
if 'search_volume' not in page_march.columns:
    print("Warning: 'search_volume' column still missing from page_march after merge. Adding as NaN.")
    page_march['search_volume'] = np.nan

print("Rows after content join:", len(page_march))
display(
    page_march[
        [
            "content_hash_id",
            "impressions",
            "clicks",
            "ctr",
            "avg_position",
            "search_volume"
        ]
    ].head()
)

Rows after content join: 176738


,content_hash_id,impressions,clicks,ctr,avg_position,search_volume
0,content_7a105f548d9c6916,6523.0,7.0,0.107313,6.893301,20
1,content_a3ea9792f793ec72,453.0,0.0,0.000000,3.214128,40
2,content_36c36abc7650d7af,5630.0,6.0,0.106572,6.535346,10
3,content_a7da352b73b02668,4944.0,13.0,0.262945,7.435680,10
4,content_1855a661b4d36130,429.0,1.0,0.233100,3.871795,50


In [23]:
# ============================================
# Signal 2 — Search-volume buckets
# ============================================

# Ensure 'search_volume' exists in page_march right before using it.
# This is a defensive measure to prevent KeyError if the column is unexpectedly missing.
if 'search_volume' not in page_march.columns:
    print("Warning: 'search_volume' column is missing from page_march in esttK6BvvR2V. Attempting to add as NaN to proceed.")
    page_march['search_volume'] = np.nan

# Attempt to create the bins without labels to get the actual number of bins.
# If page_march["search_volume"] contains too many identical values,
# qcut with duplicates="drop" will result in fewer bins than q.
cut_result = pd.qcut(
    page_march["search_volume"],
    q=4,
    duplicates="drop",
    retbins=False # No need for bins, just the categorical series
)
num_actual_bins = len(cut_result.cat.categories)

# Generate labels dynamically based on the actual number of bins
dynamic_labels = [f"Q{i+1}" for i in range(num_actual_bins)]
if num_actual_bins == 1:
    dynamic_labels[0] = "All"
elif num_actual_bins == 2:
    dynamic_labels = ["Low", "High"]
elif num_actual_bins == 3:
    dynamic_labels = ["Low", "Mid", "High"]
elif num_actual_bins == 4:
    dynamic_labels = ["Q1 Low", "Q2", "Q3", "Q4 High"]

page_march["volume_bucket"] = pd.qcut(
    page_march["search_volume"],
    q=4,
    labels=dynamic_labels,
    duplicates="drop"
)

volume_signal = (
    page_march
    .groupby("volume_bucket", observed=False)
    .agg(
        mean_impressions=("impressions", "mean"),
        median_impressions=("impressions", "median"),
        mean_ctr=("ctr", "mean"),
        impressions=("impressions", "sum"),
        clicks=("clicks", "sum"),
        n=("content_hash_id", "count")
    )
    .reset_index()
)

display(volume_signal)

,volume_bucket,mean_impressions,median_impressions,mean_ctr,impressions,clicks,n
0,Low,1693.946489,221.0,0.337849,179932690.0,521821.0,106221
1,Mid,1870.070786,270.0,0.261014,40103668.0,124719.0,21445
2,High,1768.182348,223.0,0.226322,58597563.0,166444.0,33140


In [21]:
# ============================================
# Signal 2 — Honest verdict
# ============================================

ordered_volume = (
    volume_signal
    .dropna(subset=["mean_impressions"])
    ["mean_impressions"]
    .to_numpy()
)

if len(ordered_volume) >= 3:
    increases = np.sum(np.diff(ordered_volume) > 0)
    decreases = np.sum(np.diff(ordered_volume) < 0)

    if increases >= len(ordered_volume) - 1:
        signal2_verdict = "CONFIRMED"
    elif decreases >= len(ordered_volume) - 1:
        signal2_verdict = "OPPOSITE"
    else:
        signal2_verdict = "MIXED"
else:
    signal2_verdict = "FALSE"

print("Signal 2 verdict:", signal2_verdict)

Signal 2 verdict: MIXED


### Signal 2 verdict — MIXED

**Verdict: MIXED**

The search-volume buckets do not show a consistently increasing relationship with search impressions across the March 2026 data.

Because the relationship is mixed, I will **not use search volume as a scoring feature in the baseline rule**. This prevents the rule from assigning priority based on a signal that does not show sufficiently stable evidence in this slice.

This is an intentional exclusion rather than a failed signal check: the audit showed that search volume should not be relied upon by this baseline.

## 2. Build the ranked queue

### Baseline rule

I prioritize pages that:

1. have enough impressions to represent meaningful search visibility;
2. have a relatively weak CTR;
3. are positioned in a range where improving the result could plausibly matter.

The score is intentionally transparent:

`score = visibility_flag × position_opportunity × ctr_gap`

where:

- `visibility_flag` identifies pages with meaningful impressions;
- `position_opportunity` gives priority to pages ranking in positions 4–20;
- `ctr_gap` measures how far the page's CTR is below the observed median CTR for its position bucket.

Every scored page receives exactly one reason code and one action label.

In [25]:
# ============================================
# Section 2 — Baseline score
# ============================================

# Recreate position_bucket after all merges to ensure it's present
page_march["position_bucket"] = pd.cut(
    page_march["avg_position"],
    bins=[0, 3, 10, 20, 50, np.inf],
    labels=["1-3", "4-10", "11-20", "21-50", "51+"],
    include_lowest=True
)

# Minimum visibility threshold.
# This is intentionally simple and transparent.
visibility_threshold = page_march["impressions"].quantile(0.50)

# Median CTR within each position bucket.
position_ctr_reference = (
    page_march
    .groupby("position_bucket", observed=False)["ctr"]
    .median()
    .rename("position_median_ctr")
    .reset_index()
)

page_march = page_march.merge(
    position_ctr_reference,
    on="position_bucket",
    how="left"
)

# Flags
page_march["visible"] = (
    page_march["impressions"] >= visibility_threshold
).astype(int)

page_march["position_opportunity"] = (
    page_march["avg_position"].between(4, 20, inclusive="both")
).astype(int)

# CTR gap: positive means below the bucket median.
page_march["ctr_gap"] = (
    page_march["position_median_ctr"] - page_march["ctr"]
).clip(lower=0)

# Transparent score — no fitted weights.
page_march["score"] = (
    page_march["visible"]
    * page_march["position_opportunity"]
    * page_march["ctr_gap"]
)

# One reason code and one action.
page_march["reason_code"] = np.where(
    page_march["score"] > 0,
    "visible_position_ctr_gap",
    "not_prioritized"
)

page_march["action"] = np.where(
    page_march["score"] > 0,
    "REVIEW_CTR",
    "NO_ACTION"
)

# Rank descending by score.
page_march = page_march.sort_values(
    ["score", "impressions"],
    ascending=[False, False]
).reset_index(drop=True)

page_march["rank"] = np.arange(1, len(page_march) + 1)

print(f"Visibility threshold: {visibility_threshold:,.2f}")
print(
    "Prioritized pages:",
    int((page_march["score"] > 0).sum())
)

display(
    page_march[
        [
            "rank",
            "client_hash_id",
            "content_hash_id",
            "impressions",
            "clicks",
            "ctr",
            "avg_position",
            "position_median_ctr",
            "ctr_gap",
            "score",
            "reason_code",
            "action"
        ]
    ].head(20)
)

Visibility threshold: 173.00
Prioritized pages: 0


,rank,client_hash_id,content_hash_id,impressions,clicks,ctr,avg_position,position_median_ctr,ctr_gap,score,reason_code,action
0,1,client_e547b89c05043229,content_eadb33b5df496f4a,617124.0,5668.0,0.918454,2.331470,0.0,0.0,0.0,not_prioritized,NO_ACTION
1,2,client_e547b89c05043229,content_ec2e0346994fb5a5,245276.0,1480.0,0.603402,2.757730,0.0,0.0,0.0,not_prioritized,NO_ACTION
2,3,client_23a62021009f63c4,content_e8a52cf3d5988c07,244931.0,669.0,0.273138,15.173490,0.0,0.0,0.0,not_prioritized,NO_ACTION
3,4,client_e547b89c05043229,content_0e03de7680314cd5,221310.0,720.0,0.325336,2.506100,0.0,0.0,0.0,not_prioritized,NO_ACTION
4,5,client_23a62021009f63c4,content_44f34c0a90047651,212404.0,24.0,0.011299,0.665877,0.0,0.0,0.0,not_prioritized,NO_ACTION
5,6,client_62f4a7e64f5e0096,content_7172a7fad43f0998,205867.0,862.0,0.418717,3.298139,0.0,0.0,0.0,not_prioritized,NO_ACTION
6,7,client_08a6a72ff48e62c0,content_e7b5dd4dff461ad2,205045.0,2446.0,1.192909,4.453174,0.0,0.0,0.0,not_prioritized,NO_ACTION
7,8,client_e547b89c05043229,content_8d7d99f109e19aa2,203497.0,289.0,0.142017,2.468557,0.0,0.0,0.0,not_prioritized,NO_ACTION
8,9,client_62f4a7e64f5e0096,content_f107e54b10b43725,195997.0,996.0,0.508171,3.179023,0.0,0.0,0.0,not_prioritized,NO_ACTION
9,10,client_23a62021009f63c4,content_36e53e9c707674fc,194579.0,242.0,0.124371,32.786981,0.0,0.0,0.0,not_prioritized,NO_ACTION


In [26]:
# ============================================
# Write ranked queue
# ============================================

import os

os.makedirs("work/outputs", exist_ok=True)

output_columns = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "impressions",
    "clicks",
    "ctr",
    "avg_position",
    "search_volume",
    "score",
    "reason_code",
    "action"
]

queue = page_march[output_columns].copy()

queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print(
    "Wrote:",
    "work/outputs/baseline_action_score.csv"
)

print("Rows:", len(queue))


Wrote: work/outputs/baseline_action_score.csv
Rows: 176738


## 3. Top-20 review

The top of a rule-based ranking is where weak logic becomes easiest to spot.

For each of the top 20 pages, I review:

- the action;
- the reason code;
- why the page was ranked;
- a confidence note;
- what evidence would make the recommendation wrong.

This is a skeptical human review, not a claim that every ranked page needs the recommended action.

In [27]:
# ============================================
# Top-20 review data
# ============================================

top20 = queue.head(20).copy()

top20["confidence_note"] = np.where(
    top20["score"] >= top20["score"].quantile(0.75),
    "Higher-priority within this rule because the CTR gap is relatively large.",
    "Lower-priority within the selected queue; signal is weaker than the strongest picks."
)

top20["what_would_make_it_wrong"] = (
    "The CTR gap could reflect query mix, brand/non-brand mix, "
    "seasonality, SERP features, or position measurement noise rather than "
    "a page-level content problem."
)

display(top20)


,rank,client_hash_id,content_hash_id,impressions,clicks,ctr,avg_position,search_volume,score,reason_code,action,confidence_note,what_would_make_it_wrong
0,1,client_e547b89c05043229,content_eadb33b5df496f4a,617124.0,5668.0,0.918454,2.331470,390,0.0,not_prioritized,NO_ACTION,Higher-priority within this rule because the C...,"The CTR gap could reflect query mix, brand/non..."
1,2,client_e547b89c05043229,content_ec2e0346994fb5a5,245276.0,1480.0,0.603402,2.757730,0,0.0,not_prioritized,NO_ACTION,Higher-priority within this rule because the C...,"The CTR gap could reflect query mix, brand/non..."
2,3,client_23a62021009f63c4,content_e8a52cf3d5988c07,244931.0,669.0,0.273138,15.173490,10,0.0,not_prioritized,NO_ACTION,Higher-priority within this rule because the C...,"The CTR gap could reflect query mix, brand/non..."
3,4,client_e547b89c05043229,content_0e03de7680314cd5,221310.0,720.0,0.325336,2.506100,110,0.0,not_prioritized,NO_ACTION,Higher-priority within this rule because the C...,"The CTR gap could reflect query mix, brand/non..."
4,5,client_23a62021009f63c4,content_44f34c0a90047651,212404.0,24.0,0.011299,0.665877,0,0.0,not_prioritized,NO_ACTION,Higher-priority within this rule because the C...,"The CTR gap could reflect query mix, brand/non..."
5,6,client_62f4a7e64f5e0096,content_7172a7fad43f0998,205867.0,862.0,0.418717,3.298139,10,0.0,not_prioritized,NO_ACTION,Higher-priority within this rule because the C...,"The CTR gap could reflect query mix, brand/non..."
6,7,client_08a6a72ff48e62c0,content_e7b5dd4dff461ad2,205045.0,2446.0,1.192909,4.453174,1000,0.0,not_prioritized,NO_ACTION,Higher-priority within this rule because the C...,"The CTR gap could reflect query mix, brand/non..."
7,8,client_e547b89c05043229,content_8d7d99f109e19aa2,203497.0,289.0,0.142017,2.468557,70,0.0,not_prioritized,NO_ACTION,Higher-priority within this rule because the C...,"The CTR gap could reflect query mix, brand/non..."
8,9,client_62f4a7e64f5e0096,content_f107e54b10b43725,195997.0,996.0,0.508171,3.179023,0,0.0,not_prioritized,NO_ACTION,Higher-priority within this rule because the C...,"The CTR gap could reflect query mix, brand/non..."
9,10,client_23a62021009f63c4,content_36e53e9c707674fc,194579.0,242.0,0.124371,32.786981,0,0.0,not_prioritized,NO_ACTION,Higher-priority within this rule because the C...,"The CTR gap could reflect query mix, brand/non..."


In [28]:
# ============================================
# Human-readable top-20 review
# ============================================

for _, row in top20.iterrows():

    print(
        f"{int(row['rank'])}. "
        f"Action={row['action']} | "
        f"Reason={row['reason_code']} | "
        f"Score={row['score']:.4f}"
    )

    print(
        f"   Why: visible page with position "
        f"{row['avg_position']:.2f} and CTR "
        f"{row['ctr']:.3f}% below its position-bucket reference."
    )

    print(
        f"   Confidence: {row['confidence_note']}"
    )

    print(
        f"   What would make it wrong: "
        f"{row['what_would_make_it_wrong']}"
    )

    print()

1. Action=NO_ACTION | Reason=not_prioritized | Score=0.0000
   Why: visible page with position 2.33 and CTR 0.918% below its position-bucket reference.
   Confidence: Higher-priority within this rule because the CTR gap is relatively large.
   What would make it wrong: The CTR gap could reflect query mix, brand/non-brand mix, seasonality, SERP features, or position measurement noise rather than a page-level content problem.

2. Action=NO_ACTION | Reason=not_prioritized | Score=0.0000
   Why: visible page with position 2.76 and CTR 0.603% below its position-bucket reference.
   Confidence: Higher-priority within this rule because the CTR gap is relatively large.
   What would make it wrong: The CTR gap could reflect query mix, brand/non-brand mix, seasonality, SERP features, or position measurement noise rather than a page-level content problem.

3. Action=NO_ACTION | Reason=not_prioritized | Score=0.0000
   Why: visible page with position 15.17 and CTR 0.273% below its position-bucket 

## 4. Weak picks + leakage check

### Weak-pick review

A useful baseline should have some questionable picks. I therefore inspect the lower end of the top-20 list rather than assuming every recommendation is correct.

### Leakage check

The baseline uses only:

- March 2026 GSC performance for signal auditing;
- static `dim_content` properties;
- current-slice impressions, clicks, and position for ranking.

I do not use:

- `fact_content_query_90d`;
- future-window features;
- product flags;
- a future label;
- `trend_direction`;
- `trend_pct`;
- client names, URLs, or private queries.

The final June `_sample` month is also not used.

In [29]:
# ============================================
# Weak picks
# ============================================

print("Potentially weak picks from the bottom of the top-20:\n")

weak_picks = top20.tail(5)

for _, row in weak_picks.iterrows():

    print(
        f"Rank {int(row['rank'])}: "
        f"score={row['score']:.4f}, "
        f"position={row['avg_position']:.2f}, "
        f"CTR={row['ctr']:.3f}%, "
        f"impressions={row['impressions']:,.0f}"
    )

    print(
        "Why it may be weak: the rule sees a CTR gap, "
        "but that gap does not prove that changing the page "
        "would improve performance."
    )

    print()


Potentially weak picks from the bottom of the top-20:

Rank 16: score=0.0000, position=2.91, CTR=0.615%, impressions=152,806
Why it may be weak: the rule sees a CTR gap, but that gap does not prove that changing the page would improve performance.

Rank 17: score=0.0000, position=3.43, CTR=0.270%, impressions=151,166
Why it may be weak: the rule sees a CTR gap, but that gap does not prove that changing the page would improve performance.

Rank 18: score=0.0000, position=22.14, CTR=0.042%, impressions=143,907
Why it may be weak: the rule sees a CTR gap, but that gap does not prove that changing the page would improve performance.

Rank 19: score=0.0000, position=3.17, CTR=0.030%, impressions=143,019
Why it may be weak: the rule sees a CTR gap, but that gap does not prove that changing the page would improve performance.

Rank 20: score=0.0000, position=3.29, CTR=0.241%, impressions=142,304
Why it may be weak: the rule sees a CTR gap, but that gap does not prove that changing the page wo

In [30]:
# ============================================
# Leakage / feature audit
# ============================================

forbidden_terms = [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "fact_content_query_90d",
    "product_flag",
    "future"
]

used_feature_text = """
gsc_impressions
gsc_clicks
gsc_sum_position
gsc_avg_position
search_volume
content_type
main_intent
"""

print("Leakage audit:")

for term in forbidden_terms:
    print(
        f"{term:25} ->",
        "FOUND" if term.lower() in used_feature_text.lower() else "not used"
    )

print("\nFinal month check:")
print(
    con.sql(f"""
        SELECT
            MIN(report_date) AS min_date,
            MAX(report_date) AS max_date,
            COUNT(*) AS n
        FROM {TABLES["fact_daily"]}
        WHERE month = '2026-03'
    """).df()
)

Leakage audit:
trend_direction           -> not used
trend_pct                 -> not used
is_declining_label        -> not used
fact_content_query_90d    -> not used
product_flag              -> not used
future                    -> not used

Final month check:
    min_date   max_date        n
0 2026-03-01 2026-03-31  9841378


## Baseline summary

This notebook produced a transparent, hand-written baseline rather than a fitted model.

### Signals

- **CTR vs. position:** verdict is printed from the observed bucket pattern above.
- **Search volume:** verdict is printed from the observed bucket pattern above.

### Rule

Pages receive priority when they have meaningful visibility, rank in positions 4–20, and have CTR below the median observed for their position bucket.

### Output

The ranked queue is written to:

`work/outputs/baseline_action_score.csv`

The CSV is intentionally generated by the notebook and should not be committed to Git.

### Limitation

The ranking is a decision-support baseline, not proof that changing a page will improve performance. The top-20 review explicitly records what could make each recommendation wrong.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.